# Part B - NumPy
All 10 tasks on the weather grid dataset. No loops used except in B5, where a loop is required.

In [1]:
import numpy as np

np.random.seed(7)
cities = np.array(["Kathmandu", "Pokhara", "Biratnagar", "Lalitpur", "Birgunj"])
days = np.array([f"D{i:02d}" for i in range(1, 15)])
temps = np.random.randint(14, 36, size=(5, 14)).astype(float)
humidity = np.random.randint(40, 95, size=(5, 14)).astype(float)
temps[2, 5] = np.nan
temps[4, 11] = np.nan

## B1 - Array anatomy and reshaping

In [2]:
print("B1 - shape:", temps.shape)
print("B1 - ndim:", temps.ndim)
print("B1 - dtype:", temps.dtype)
print("B1 - size:", temps.size)

flat = np.arange(70)
reshaped_a = flat.reshape(5, 14)
reshaped_b = flat.reshape(5, -1)
print("B1 - both reshapes same:", np.array_equal(reshaped_a, reshaped_b))

back_to_flat = reshaped_a.reshape(-1)
print("B1 - flattened back matches original:", np.array_equal(back_to_flat, flat))

# -1 means "figure out this dimension yourself so the total count still matches".
# reshape does not copy data, it just looks at the same numbers in memory with
# a different shape on top, which is why it is fast and memory friendly.

B1 - shape: (5, 14)
B1 - ndim: 2
B1 - dtype: float64
B1 - size: 70
B1 - both reshapes same: True
B1 - flattened back matches original: True


## B2 - Indexing and slicing

In [3]:
a = temps[1, 2]
print("B2a - Pokhara day 3:", a, "shape:", np.shape(a))

b = temps[:, -1]
print("B2b - every city on last day:", b, "shape:", b.shape)

c = temps[2, :]
print("B2c - Biratnagar full fortnight:", c, "shape:", c.shape)

d = temps[0:3, 3:8]
print("B2d - first 3 cities, days 4-8:\n", d, "shape:", d.shape)

e = temps[3, ::2]
print("B2e - Lalitpur every other day:", e, "shape:", e.shape)

f = temps[0, ::-1]
print("B2f - Kathmandu reversed:", f, "shape:", f.shape)

B2a - Pokhara day 3: 14.0 shape: ()
B2b - every city on last day: [30. 28. 21. 18. 30.] shape: (5,)
B2c - Biratnagar full fortnight: [20. 18. 28. 23. 17. nan 22. 33. 30. 15. 14. 30. 26. 21.] shape: (14,)
B2d - first 3 cities, days 4-8:
 [[33. 21. 28. 22. 28.]
 [25. 20. 33. 26. 19.]
 [23. 17. nan 22. 33.]] shape: (3, 5)
B2e - Lalitpur every other day: [24. 17. 21. 35. 17. 17. 33.] shape: (7,)
B2f - Kathmandu reversed: [30. 18. 20. 21. 22. 24. 28. 22. 28. 21. 33. 17. 18. 29.] shape: (14,)


## B3 - Boolean masks

In [4]:
mask = temps > 30
print("B3a - readings above 30 per city:", mask.sum(axis=1))

hot_days_kathmandu = days[mask[0]]
print("B3b - Kathmandu days above 30:", hot_days_kathmandu)

# A mask is just a same-shaped array of True/False values that says which
# spots pass a condition. temps[temps > 30] cannot stay 2D because different
# rows can have a different number of matching values, so numpy just gives
# back a flat 1D list of the matching numbers.

B3a - readings above 30 per city: [1 2 1 2 6]
B3b - Kathmandu days above 30: ['D04']


## B4 - Views vs copies

In [5]:
city0_slice = temps[0, :3]
city0_slice[0] = 999
print("B4 - original after changing the VIEW (it changed too):", temps[0, :3])

temps[0, 0] = 20.0   # put the value back before the next demo

city0_copy = temps[0, :3].copy()
city0_copy[0] = 999
print("B4 - original after changing the COPY (unchanged):", temps[0, :3])

# Slicing gives a view by default (same memory, no copy) because it is fast.
# The danger is: if you forget it's a view and edit it, you accidentally
# edit the original array too, which can create hard to find bugs.

B4 - original after changing the VIEW (it changed too): [999.  18.  17.]
B4 - original after changing the COPY (unchanged): [20. 18. 17.]


## B5 - Vectorization vs a loop (only loop allowed in Part B)

In [6]:
import time

def to_fahrenheit_loop(arr):
    out = np.empty_like(arr)
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            out[i, j] = arr[i, j] * 9 / 5 + 32
    return out

loop_result = to_fahrenheit_loop(temps)
vector_result = temps * 9 / 5 + 32
print("B5 - loop and vectorized give same answer:", np.allclose(loop_result, vector_result, equal_nan=True))

big_array = np.random.rand(1_000_000) * 40

def to_fahrenheit_loop_1d(arr):
    out = np.empty_like(arr)
    for i in range(arr.shape[0]):
        out[i] = arr[i] * 9 / 5 + 32
    return out

start = time.perf_counter()
big_loop = to_fahrenheit_loop_1d(big_array)
mid = time.perf_counter()
big_vector = big_array * 9 / 5 + 32
end = time.perf_counter()

loop_time = mid - start
vector_time = end - mid
print(f"B5 - loop time: {loop_time:.4f}s")
print(f"B5 - vectorized time: {vector_time:.6f}s")
print(f"B5 - speedup: ~{loop_time / vector_time:.0f}x faster")

# The loop spends most of its time on python overhead - checking types and
# running python bytecode one value at a time. The vectorized version does
# the whole operation in fast compiled C code in one shot, so it barely
# takes any time at all.

B5 - loop and vectorized give same answer: True


B5 - loop time: 0.5842s
B5 - vectorized time: 0.005212s
B5 - speedup: ~112x faster


## B6 - Broadcasting

In [7]:
city_avg = np.nanmean(temps, axis=1, keepdims=True)
anomaly = temps - city_avg
print("B6a - anomaly shape:", anomaly.shape)

day_correction = np.linspace(-1, 1, 14)
corrected = temps + day_correction
print("B6b - corrected shape:", corrected.shape)

try:
    temps + np.array([1, 2, 3, 4, 5])
except ValueError as e:
    print("B6c - got this error:", e)

# Broadcasting lines shapes up starting from the RIGHT side.
# temps is (5, 14). day_correction is (14,), which matches the last axis (14),
# so it can stretch across all 5 rows - that's why B6b worked.
# The array [1,2,3,4,5] has shape (5,), which lines up against 14, not 5,
# and 5 != 14 and neither side is 1, so numpy cannot line them up and raises
# an error.

B6a - anomaly shape: (5, 14)
B6b - corrected shape: (5, 14)
B6c - got this error: operands could not be broadcast together with shapes (5,14) (5,) 


## B7 - Matrix multiplication

In [8]:
quantities = np.array([2, 3, 1])
prices = np.array([120, 90, 250])

print("B7a - * (multiply each item by item):", quantities * prices)
print("B7a - @ (dot product / total cost):", quantities @ prices)
# * just multiplies matching positions and keeps them separate.
# @ multiplies matching positions AND adds them up into one single number,
# which is what we actually want for a total bill.

mean_temp_per_day = np.nanmean(temps, axis=0)
mean_humidity_per_day = np.nanmean(humidity, axis=0)
features = np.column_stack([mean_temp_per_day, mean_humidity_per_day])
weights = np.array([0.6, -0.1])

comfort_score = features @ weights
print("B7b - features shape:", features.shape)
print("B7b - weights shape:", weights.shape)
print("B7b - comfort score shape:", comfort_score.shape)

# features has 2 columns and weights has 2 values - these "inner" numbers
# have to match for @ to work, because each row's 2 numbers get combined
# with the 2 weights into a single score for that day.

B7a - * (multiply each item by item): [240 270 250]
B7a - @ (dot product / total cost): 760
B7b - features shape: (14, 2)
B7b - weights shape: (2,)
B7b - comfort score shape: (14,)


## B8 - Aggregating along an axis

In [9]:
print("B8 - mean per city:", np.nanmean(temps, axis=1))
print("B8 - max per city:", np.nanmax(temps, axis=1))
print("B8 - min per city:", np.nanmin(temps, axis=1))
print("B8 - std per city:", np.nanstd(temps, axis=1))

print("B8 - mean per day:", np.nanmean(temps, axis=0))
print("B8 - max per day:", np.nanmax(temps, axis=0))

hottest_day_index = np.nanargmax(temps, axis=1)
print("B8 - hottest day per city:", days[hottest_day_index])

# axis=1 collapses across the days, so we are left with one number per CITY.
# axis=0 collapses across the cities, so we are left with one number per DAY.

B8 - mean per city: [23.         23.57142857 22.84615385 20.07142857 26.53846154]
B8 - max per city: [33. 35. 33. 35. 35.]
B8 - min per city: [17. 14. 14. 14. 15.]
B8 - std per city: [4.73588127 6.41108733 5.89453663 6.41943796 7.11028924]
B8 - mean per day: [23.6  20.   19.   23.6  22.   22.75 27.6  26.   27.6  23.4  16.6  20.25
 25.8  25.4 ]
B8 - max per day: [33. 26. 28. 33. 31. 33. 35. 33. 35. 35. 21. 30. 33. 30.]
B8 - hottest day per city: ['D04' 'D09' 'D08' 'D07' 'D10']


## B9 - Reproducible randomness and a train/test split

In [10]:
flat_temps = temps.flatten()

np.random.seed(42)
index = np.arange(len(flat_temps))
np.random.shuffle(index)

split_point = int(len(index) * 0.8)
train_index = index[:split_point]
test_index = index[split_point:]

print("B9 - train shape:", train_index.shape)
print("B9 - test shape:", test_index.shape)
print("B9 - sizes add up to 70:", len(train_index) + len(test_index) == 70)
print("B9 - no index shared between train and test:", len(np.intersect1d(train_index, test_index)) == 0)

np.random.seed(42)
index2 = np.arange(len(flat_temps))
np.random.shuffle(index2)
train_index2 = index2[:split_point]
test_index2 = index2[split_point:]

same_split = np.array_equal(train_index, train_index2) and np.array_equal(test_index, test_index2)
print("B9 - same seed gives same split:", same_split)

# We shuffle before splitting so the first 80% is not just whatever
# happened to be at the start of the array - it becomes a random mix.
# The seed matters because it makes the "random" shuffle repeatable, so
# someone else running the same code gets the exact same split as us.

B9 - train shape: (56,)
B9 - test shape: (14,)
B9 - sizes add up to 70: True
B9 - no index shared between train and test: True
B9 - same seed gives same split: True


## B10 - np.where, argmax, argmin

In [11]:
labels_grid = np.where(temps > 30, "hot", np.where(temps >= 22, "mild", "cold"))
print("B10a - label grid shape:", labels_grid.shape)

scores = np.array([0.08, 0.62, 0.21, 0.09])
weather_labels = ["rain", "clear", "cloudy", "fog"]

predicted_label = weather_labels[np.argmax(scores)]
print("B10b - predicted label:", predicted_label)

# max would just tell us the highest SCORE (0.62), but a classifier needs to
# know WHICH label that score belongs to. argmax gives us the position of
# the highest score, so we can look up the matching label with it.

B10a - label grid shape: (5, 14)
B10b - predicted label: clear
